# 01 — CardSim calibration probe

Pilot simulation used to verify the local CardSim setup and derive the sizing parameters for `scripts/build_dataset.py`.

1. **Verify offline DCPC loading** using the local CSVs bundled with `cardsim` (the Atlanta Fed remote endpoint breaks default downloads).
2. **Measure $\hat{f}$** (transactions per card-day), which CardSim does not expose as an input, and derive `n_days` and `payer_payee_factor` for the base dataset.
3. **Inspect the 13 raw columns** normalized into the 11-column canonical schema by `fraud_stat_engine.schema.normalize_cardsim_df()`.

*Out of scope:* EDA on the full `cardsim_base.parquet` dataset (`02_eda_cardsim_base.ipynb`).

In [30]:
import math
from pathlib import Path

import cardsim
import pandas as pd
from cardsim import Cardsim

DCPC_DIR = str(Path(cardsim.__file__).resolve().parent / "dcpc")

SEED = 42
FRAUD_RATE = 0.01
FRAUD_FLAG_THRESHOLD = 0.01
N_PAYERS = 500
N_DAYS = 30
START_DATE = "2024-01-01"

sim = Cardsim(
    seed=SEED,
    fraud_rate=FRAUD_RATE,
    fraud_flag_threshold=FRAUD_FLAG_THRESHOLD,
    dcpc_folder=DCPC_DIR,
)
df = sim.simulate(n_payers=N_PAYERS, n_days=N_DAYS, start_date=START_DATE)

print(f"Shape: {df.shape}")
df.head(3)

Sourcing DCPC data
Sourcing successful; formatting data
DCPC data sourcing and formatting complete
Starting phase one: generating simulator world


Generated payment distributions

Generating payer profiles for 500 payers
Generated payer profiles

Generating payee profiles for 50 payees
Generated payee profiles

Calculated distance matrix

Generated world in 0 seconds

Starting phase two: generating transactions within world

Generated baseline transactions

Generated card type attribute

Generated location type attribute

Generated transaction amount

Generated and added payee distance
Generated transaction time

Generated fraud flag

Generated transactions in 0 seconds

Simulator completed in 0 seconds

Shape: (14583, 13)


,run_id,day_index,date,time_seconds,hour,date_time,payer_id,payee_id,credit_card,remote,distance,amount,fraud
0,S42P500D30,0,2024-01-01,62077,17,2024-01-01 17:14:37,0,29,0,1,60.415230,12.660533,0
1,S42P500D30,0,2024-01-01,72843,20,2024-01-01 20:14:03,2,18,0,1,82.807007,48.286849,0
2,S42P500D30,0,2024-01-01,25697,7,2024-01-01 07:08:17,2,16,0,1,110.113579,6.235227,0


## 1. Probe results

Each quantity is tagged by origin: **decided** (input), **measured** (simulator output), or **derived** (formula). The observed fraud rate (~1.00%) is **decided**, not free: `fraud_flag_threshold = 0.01` labels the top 1% of rows by risk score.

In [31]:
n_merchants = df["payee_id"].nunique()

probe_summary = pd.DataFrame(
    [
        ("Cards simulated",    f"{N_PAYERS}",               "Decided (n_payers)"),
        ("Days simulated",     f"{N_DAYS}",                 f"Decided (n_days); {N_PAYERS * N_DAYS:,} card-days"),
        ("Unique merchants",   f"{n_merchants}",            f"Derived ({N_PAYERS}/{N_PAYERS / n_merchants:g})"),
        ("Total transactions", f"{len(df):,}",              "Measured"),
        ("Fraud rate",         f"{df['fraud'].mean():.4%}", "Decided (fraud_flag_threshold = 0.01)"),
        ("Raw columns",        f"{df.shape[1]}",            "CardSim internal schema"),
    ],
    columns=["Quantity", "Value", "Origin"],
).set_index("Quantity")

probe_summary

,Value,Origin
Quantity,,
Cards simulated,500,Decided (n_payers)
Days simulated,30,"Decided (n_days); 15,000 card-days"
Unique merchants,50,Derived (500/10)
Total transactions,"14,583",Measured
Fraud rate,1.0012%,Decided (fraud_flag_threshold = 0.01)
Raw columns,13,CardSim internal schema


## 2. Sizing `scripts/build_dataset.py`

$$\hat{f} = \frac{\text{transactions}}{n_{\text{cards}} \times n_{\text{days}}}, \qquad n_{\text{days}} = \left\lceil \frac{N_{\text{target}}}{n_{\text{cards}} \cdot \hat{f}} \right\rceil, \qquad \text{payer\_payee\_factor} = \frac{n_{\text{cards}}}{n_{\text{merchants}}}$$

In [32]:
# Measured from probe
f_hat = len(df) / (N_PAYERS * N_DAYS)

# Decided targets for cardsim_base.parquet
TARGET_TX = 200_000
BUILD_N_PAYERS = 17_500
BUILD_N_MERCHANTS = 4_000

# Derived inputs for scripts/build_dataset.py
BUILD_N_DAYS = math.ceil(TARGET_TX / (BUILD_N_PAYERS * f_hat))
BUILD_PAYER_PAYEE_FACTOR = BUILD_N_PAYERS / BUILD_N_MERCHANTS
EXPECTED_TX = round(BUILD_N_PAYERS * BUILD_N_DAYS * f_hat)

build_config = pd.DataFrame(
    [
        ("f_hat (tx/card/day)", f"{f_hat:.4f}",                "Measured"),
        ("target_tx",           f"{TARGET_TX:,}",              "Decided"),
        ("n_payers",            f"{BUILD_N_PAYERS:,}",         "Decided"),
        ("n_merchants",         f"{BUILD_N_MERCHANTS:,}",      "Decided"),
        ("n_days",              f"{BUILD_N_DAYS}",             "Derived (ceil)"),
        ("payer_payee_factor",  f"{BUILD_PAYER_PAYEE_FACTOR}", "Derived (n_payers / n_merchants)"),
        ("expected_tx",         f"~{EXPECTED_TX:,}",           "Derived (n_payers * n_days * f_hat)"),
    ],
    columns=["Parameter", "Value", "Category"],
).set_index("Parameter")

build_config

,Value,Category
Parameter,,
f_hat (tx/card/day),0.9722,Measured
target_tx,"200,000",Decided
n_payers,"17,500",Decided
n_merchants,"4,000",Decided
n_days,12,Derived (ceil)
payer_payee_factor,4.375,Derived (n_payers / n_merchants)
expected_tx,"~204,162",Derived (n_payers * n_days * f_hat)


## 3. Raw schema inventory (input to `schema.py`)

Snapshot of the 13 raw columns before `normalize_cardsim_df()` renames entities, maps binary flags (`credit_card`, `remote`) to enums, and drops redundant time fields (`run_id`, `day_index`, `date`, `time_seconds`, `hour`).

In [33]:
pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "nulls": df.isna().sum(),
        "sample": df.iloc[0],
    }
)

,dtype,nulls,sample
run_id,str,0,S42P500D30
day_index,int64,0,0
date,datetime64[us],0,2024-01-01 00:00:00
time_seconds,int64,0,62077
hour,int32,0,17
date_time,datetime64[us],0,2024-01-01 17:14:37
payer_id,int64,0,0
payee_id,int64,0,29
credit_card,int64,0,0
remote,int64,0,1
